# Task 4: Forecasting Access and Usage (2025-2027) — Ethiopia Financial Inclusion

**Author:** Sosina Ayele
**Program:** 10 Academy — Week 11 Challenge
**Date:** July 2026

This notebook forecasts Account Ownership (Access) and a proxy Usage
indicator for 2025-2027 using a trend-regression baseline, an event-
augmented model (baseline + Task 3's estimated event effects), and three
scenarios (optimistic / base / pessimistic). Reusable logic lives in
`src/forecasting.py`.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_unified_data, split_by_record_type
from analysis import get_indicator_series
from forecasting import fit_trend_model, forecast_baseline, apply_event_augmentation, build_scenarios

sns.set_style('whitegrid')

In [ ]:
df = load_unified_data('../data/processed/ethiopia_fi_enriched.csv')
parts = split_by_record_type(df)
obs, events, targets = parts['observations'], parts['events'], parts['targets']

## 1. Define Targets

- **Access:** `ACC_OWNERSHIP` — % of adults with an account at a financial institution or via mobile money.
- **Usage:** `ACC_MM_ACCOUNT` used as the best-covered available Usage-adjacent proxy in this dataset
  (mobile money account ownership); a dedicated `USG_DIGITAL_PAYMENT` series was not present with
  enough multi-year coverage to fit a standalone trend model — this is documented as a limitation below.

## 2. Access Forecast — Baseline Trend

In [ ]:
acc = get_indicator_series(obs, 'ACC_OWNERSHIP')
print(acc[['observation_date', 'value_numeric']])

acc_fit = fit_trend_model(acc)
print(f"\nTrend: {acc_fit['model'].params['years_since_start']:.3f} pp/year "
      f"(R-squared: {acc_fit['model'].rsquared:.3f})")

acc_baseline = forecast_baseline(acc_fit, [2025, 2026, 2027])
print(acc_baseline)

**Note on R-squared with n=6:** with only 6 historical points, a high
R-squared should not be read as strong statistical confidence — it mainly
reflects that Ethiopia's account-ownership growth has been fairly monotonic,
not that the model is well-validated out-of-sample. Confidence intervals
below are correspondingly wide.

## 3. Event-Augmented Forecast

In [ ]:
# Cumulative event effects on ACC_OWNERSHIP, translated from the Task 3
# association matrix (impact_score units) into approximate pp terms using
# each event's stated magnitude tier as a rough pp scale:
#   low ~ 0.5pp, medium ~ 1.5pp, high ~ 3pp (cumulative, once fully ramped)
# This is an explicit, documented simplification pending a full regression
# calibration in a future iteration.
event_effects_acc = {
    2025: 1.5,   # NFIS-II (medium, partially ramped by 2025) + Fayda (partial ramp)
    2026: 2.5,   # NFIS-II fully ramped, Fayda fully ramped
    2027: 2.5,   # held constant once fully ramped (no new Access-relevant events beyond 2025 in current catalog)
}

acc_augmented = apply_event_augmentation(acc_baseline, event_effects_acc)
print(acc_augmented)

## 4. Scenario Analysis

In [ ]:
acc_scenarios = build_scenarios(acc_augmented, optimistic_multiplier=1.6, pessimistic_multiplier=0.4)
print(acc_scenarios)

plt.figure(figsize=(10, 6))
plt.plot(acc['observation_date'].dt.year, acc['value_numeric'], marker='o', color='black', label='Observed')
plt.plot(acc_scenarios['year'], acc_scenarios['base'], marker='o', color='#1B9AAA', label='Base (event-augmented)')
plt.fill_between(acc_scenarios['year'], acc_scenarios['pessimistic'], acc_scenarios['optimistic'],
                  color='#1B9AAA', alpha=0.2, label='Optimistic-Pessimistic range')
plt.fill_between(acc_scenarios['year'], acc_scenarios['ci_lower'], acc_scenarios['ci_upper'],
                  color='gray', alpha=0.15, label='Trend-only 95% CI')
plt.axhline(60, linestyle='--', color='darkred', alpha=0.7, label='60% inclusion goal')
plt.title('Ethiopia Account Ownership Forecast, 2025-2027')
plt.ylabel('% of adults')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/access_forecast_scenarios.png', dpi=150)
plt.show()

## 5. Usage Proxy Forecast (Mobile Money Account Ownership)

In [ ]:
mm = get_indicator_series(obs, 'ACC_MM_ACCOUNT')
print(mm[['observation_date', 'value_numeric']])

if len(mm) >= 2:
    mm_fit = fit_trend_model(mm)
    mm_baseline = forecast_baseline(mm_fit, [2025, 2026, 2027])

    if mm_baseline['ci_lower'].isna().any():
        print("\nNote: confidence intervals are undefined (NaN) because only "
              f"{len(mm)} historical points are available, leaving zero residual "
              "degrees of freedom for OLS to estimate uncertainty from. The point "
              "forecast is still computed, but should be treated as a single trend "
              "line, not a statistically bounded estimate, until more Usage-pillar "
              "survey waves are available.")

    event_effects_mm = {2025: 2.0, 2026: 3.0, 2027: 3.0}  # M-Pesa + EthioPay + M-Pesa Interop, ramped
    mm_augmented = apply_event_augmentation(mm_baseline, event_effects_mm)
    mm_scenarios = build_scenarios(mm_augmented, optimistic_multiplier=1.6, pessimistic_multiplier=0.4)
    print(mm_scenarios)
else:
    print("Fewer than 2 ACC_MM_ACCOUNT observations available — cannot fit a trend model.")
    print("This is a direct consequence of the sparse Usage-pillar coverage flagged in Task 2.")

## 6. Forecast Table Summary

In [ ]:
summary = acc_scenarios.copy()
summary['indicator'] = 'ACC_OWNERSHIP'
if len(mm) >= 2:
    mm_summary = mm_scenarios.copy()
    mm_summary['indicator'] = 'ACC_MM_ACCOUNT'
    summary = pd.concat([summary, mm_summary], ignore_index=True)

summary = summary[['indicator', 'year', 'forecast', 'ci_lower', 'ci_upper', 'pessimistic', 'base', 'optimistic']]
print(summary.round(1))
summary.to_csv('../reports/forecast_summary.csv', index=False)

## 7. Interpretation

- **Access (Account Ownership):** the base scenario projects continued but
  modest growth toward the low-to-mid 50s% by 2027, still short of the
  60% inclusion goal referenced in the dashboard requirements. The
  optimistic scenario (NFIS-II and Fayda effects landing faster/larger
  than estimated) approaches the high 50s%; the pessimistic scenario shows
  Access essentially flat relative to 2024, consistent with the
  deceleration pattern already observed 2021-2024.
- **Events with the largest potential impact:** NFIS-II (long lag, but the
  only event with an explicit numeric policy target attached) and Fayda
  digital ID (infrastructure, medium magnitude, 24-month lag) dominate the
  Access-side event-augmentation; M-Pesa, EthioPay, and M-Pesa Interop
  dominate the Usage-side proxy.
- **Key uncertainties:** (1) only 6 historical Access points and as few as 2 Usage
  points underlie these trend fits — Access has a usable confidence interval, but
  Usage indicators with only 2 points have zero residual degrees of freedom, so
  their confidence intervals are undefined (NaN) and the point forecast should be
  read as a single trend line rather than a statistically bounded estimate;
  (2) event effects are currently mapped from ordinal impact_score tiers to pp
  values using a documented but simplified rule-of-thumb (low/medium/high →
  ~0.5/1.5/3pp), not a fitted regression coefficient — refining this mapping with
  more Ethiopian pre/post data points (as more Findex waves or NBE reports
  become available) is the clearest next step to tighten these forecasts.